<a href="https://colab.research.google.com/github/ernestoaguaysol-unpaz/sistemas-inteligentes-2026/blob/main/02_redes_neuronales/021_RNA_clasificacion_vinos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import seaborn as sns
sns.set_style("darkgrid")
import matplotlib.pyplot as plt
import pandas as pd
import tensorflow as tf

from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Carga del dataset.
dataset = pd.read_csv('dataset_vinos.csv')
dataset.head()

In [ ]:
# Mostrar gráfico.
sns.scatterplot(data=dataset, x='alcohol', y='intensidad_color', hue='clase')

In [ ]:
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

In [ ]:
y

In [ ]:
# Convertimos las etiquetas de clase a one-hot encoding
# Restamos 1 para que las clases comiencen desde 0 (1,2,3 -> 0,1,2)
y = tf.keras.utils.to_categorical(y-1, num_classes=3)

In [ ]:
y

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 0)

# Escalado de características
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
modelo = tf.keras.Sequential([
    tf.keras.layers.Input(shape=X_train.shape[1:]),
    tf.keras.layers.Dense(5, activation='tanh'),
    tf.keras.layers.Dense(5, activation='tanh'),
    tf.keras.layers.Dense(3, activation='softmax')
])

In [ ]:
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss=tf.keras.losses.categorical_crossentropy,
    metrics=["accuracy"]
)

In [ ]:
historial = modelo.fit(X_train, y_train, epochs=50, verbose=0)

In [ ]:
print(historial.history.keys())

In [ ]:
plt.title('Historial de rendimiento Exactitud')
plt.plot(historial.history['accuracy'], label="Exactitud")
plt.xlabel('Epochs')
plt.ylabel('Rendimiento Exactitud')
plt.legend()
plt.show()

In [ ]:
plt.title('Historial de pérdida')
plt.plot(historial.history['loss'], label="Pérdida")
plt.xlabel('Epochs')
plt.ylabel('Pérdida')
plt.legend()
plt.show()

In [ ]:
# Evaluar el modelo con conjunto de entrenamiento
modelo.evaluate(X_train, y_train)

In [ ]:
# Evaluar el modelo con conjunto de prueba
modelo.evaluate(X_test, y_test)

In [ ]:
# Hacer predicciones
y_pred = modelo.predict(X_test)

In [ ]:
y_pred[:10]

In [ ]:
y_pred_class = np.round(y_pred)
y_pred_class[:10]

In [ ]:
from sklearn.metrics import confusion_matrix
y_pred_class = np.argmax(y_pred_class, axis=1)
y_test_class = np.argmax(y_test, axis=1)
confusion_matrix(y_test_class, y_pred_class)

In [ ]:
# Matriz de confusión
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test_class, y_pred_class)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=[1, 2, 3], yticklabels=[1, 2, 3])
plt.title('Matriz de Confusión')
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.show()

In [ ]:
# Plotear la matriz de confusión con mapas de carlo de seaborn
import seaborn as sns

sns.set(font_scale=2)

def plot_matriz_de_confusion(matriz_de_confusion):

    fig, ax = plt.subplots(figsize=(8, 8))

    ax = sns.heatmap(
        matriz_de_confusion,
        annot=True,
        cbar=False,
        fmt='d'
    )

    plt.suptitle("Matriz de confusión")
    plt.xlabel("Predicción")
    plt.ylabel("Etiqueta real")


matriz_de_confusion = confusion_matrix(y_test_class, y_pred_class)
plot_matriz_de_confusion(matriz_de_confusion)